# 02 · Exploratory analysis

Questions about tree cover loss in India, 2001–2025. Every question calls a function in `src/ledger/explore.py`, so the logic is reusable and tested. Results are written up in `docs/eda_memo.md`.

**Ground rules**
- Always filter to one canopy threshold (30%) before summing. Each hectare appears once per threshold it clears, so summing across thresholds double counts.
- Data: GFW v20260424 (loss) and v20251202 (fire alerts), GADM 4.1 districts. 2025 is provisional.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ledger import explore as ex
from ledger.config import load_settings
from ledger.frames import districts, read_raw

s = load_settings()
chg, summ = read_raw(s, "tcl_change"), read_raw(s, "tcl_summary")
viirs, names = read_raw(s, "viirs_weekly"), districts(read_raw(s, "boundaries"))
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

## 1 · Is loss rising or falling?
**Look for:** the step up around 2013–2016. The UMD method and satellite inputs changed in 2011 and 2015, so part of any rise across those years is better detection, not more clearing.

**Found:** loss peaked in 2017 (179 kha). The 2001–2011 average was 62 kha, and the 2016–2020 average 143 kha. Compare years on the same side of a method break.

In [ ]:
trend = ex.national_trend(chg)
ax = (trend["loss_ha"] / 1000).plot(marker="o", lw=1, label="annual loss")
(trend["loss_ma3"] / 1000).plot(ax=ax, lw=2.5, label="3-year moving average")
for y in ex.METHOD_BREAKS:
    ax.axvline(y - 0.5, color="grey", ls="--", lw=1)
ax.axvspan(2000.5, 2014.5, color="grey", alpha=0.08)
ax.set(title="India: tree cover loss, 30% canopy", ylabel="thousand ha", xlabel="")
ax.legend()
plt.show()
trend.round(1)

In [ ]:
ex.period_compare(trend, (2016, 2020), (2021, 2025))

## 2 · Where is it happening?
**Look for:** two different rankings. Total hectares favours big forested states; loss rate (loss as a share of 2000 tree cover) shows where forest is disappearing fastest.

**Found:** the table below is 2015–2025. Mizoram leads by hectares (19% of national loss). Over 2001–2025, Tripura leads by rate (23.5% of its 2000 tree cover), then Nagaland and Mizoram. The Northeast holds about three quarters of national loss.

In [ ]:
states = ex.by_region(chg, summ, names, "state", years=(2015, 2025))
top = states.head(15).iloc[::-1]
ax = (top.set_index("state")["loss_ha"] / 1000).plot.barh(color="#5d8a72")
ax.set(title="Tree cover loss 2015–2025 by state", xlabel="thousand ha", ylabel="")
plt.show()
states.head(15)[["state", "loss_ha", "share_of_national_pct", "loss_rate_pct", "tree_cover_2000_pct"]].round(
    2
)

In [ ]:
dist = ex.by_region(chg, summ, names, "district", years=(2015, 2025))
p = ex.pareto(dist)
ax = p["cum_share_pct"].plot(lw=2)
ax.axhline(50, ls=":", color="grey")
ax.axhline(80, ls=":", color="grey")
ax.set(title="Cumulative share of loss by district (ranked)", xlabel="districts", ylabel="% of national loss")
plt.show()
print(
    ex.regions_for_share(dist, 50),
    "districts make up half of all loss;",
    ex.regions_for_share(dist, 80),
    "make up 80%, out of",
    (dist.loss_ha > 0).sum(),
)
dist.head(20)[["state", "district", "loss_ha", "loss_rate_pct"]].round(1)

## 3 · What is driving it?
**Look for:** how the mix differs by state, and how much loss falls in classes that count as deforestation. Only permanent agriculture, hard commodities, settlements and infrastructure count, plus shifting cultivation inside primary forest (Global Forest Review rule; becomes `is_deforestation` in the warehouse).

**Found:** shifting cultivation is 60% of loss nationally and 73% in the Northeast; about 40% of loss is deforestation by that rule. `Unknown` is a real label here (1% of hectares, 21% of rows).

In [ ]:
mix = ex.driver_mix(chg)
ax = mix.set_index("driver")["share_pct"].sort_values().plot.barh(color="#b07a3c")
ax.set(title="Share of loss by dominant driver, 2001–2025", xlabel="%", ylabel="")
plt.show()

# cell 7: driver mix over time
dy = (
    ex.at_threshold(chg)
    .assign(driver=lambda d: d.driver.fillna("(no driver)"))
    .pivot_table(index="year", columns="driver", values="loss_ha", aggfunc="sum")
    / 1000
)
dy.plot.area(stacked=True, lw=0, alpha=0.85, title="Loss by driver (thousand ha)").legend(
    bbox_to_anchor=(1, 1)
)
plt.show()

# cell 8: the mix differs by state
ex.driver_mix(chg, by="adm1").merge(names[["adm1", "state"]].drop_duplicates(), on="adm1").query(
    "share_pct > 25"
).sort_values(["state", "share_pct"], ascending=[True, False])

## 4 · Primary forest and protected areas
**Found:** 15% of loss is in primary forest and 0.4% is in protected areas. `is_protected` means any IUCN category, so this is a share of rows that are inside a mapped protected area.

In [ ]:
c = ex.at_threshold(chg)
pd.DataFrame(
    {
        "primary forest": c.groupby("is_primary")["loss_ha"].sum(),
        "protected area": c.groupby("is_protected")["loss_ha"].sum(),
    }
).T.assign(share_true_pct=lambda d: 100 * d[True] / (d[True] + d[False])).round(1)

## 5 · When do fires happen?
High-confidence VIIRS alerts at the 30% threshold, by month and year (2012 onward). **Found:** a spring peak, with February to April about 82% of alerts.

In [ ]:
m = ex.fire_seasonality(viirs, confidence="h", threshold=30)  # use the threshold you chose in 7.3
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(m.values, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(len(m.index)), [pd.Timestamp(2000, i, 1).strftime("%b") for i in m.index])
ax.set_xticks(range(len(m.columns)), m.columns, rotation=90)
ax.set_title("High-confidence VIIRS fire alerts by month")
fig.colorbar(im)
plt.show()

### Do alerts track fire-driven loss?
District-year pairs from 2012. **Found:** weak rank correlation (Spearman 0.29 with fire loss, 0.23 with total loss). VIIRS alerts include crop-residue burning that does not kill trees, so alerts are not a proxy for fire loss. Fire is only 1.6% of loss hectares.

In [ ]:
a = (
    viirs[(viirs.confidence == "h") & (viirs.threshold == 30)]
    .groupby(["adm1", "adm2", "year"])["alerts"]
    .sum()
)
loss = (
    ex.at_threshold(chg)
    .query("year >= 2012")
    .groupby(["adm1", "adm2", "year"])[["fire_loss_ha", "loss_ha"]]
    .sum()
)
j = loss.join(a, how="inner")
print(j.corr(method="spearman").round(2))
j.plot.scatter(
    "alerts",
    "fire_loss_ha",
    s=6,
    alpha=0.4,
    logx=True,
    logy=True,
    title="District-years: alerts vs fire-driven loss (log-log)",
)
plt.show()

## 6 · How much does the definition of "tree" matter?
Total loss at each canopy threshold. **Found:** loss at 10% is 8% higher than at 30%; at 75% it is 48% lower. Headline stays at 30% to match GFW.

In [ ]:
ex.threshold_sensitivity(chg).round(0)

## 7 · Emissions
Gross CO₂e from the loss. **Found:** 1.36 Gt for 2001–2025 (1.29 Gt for 2001–2024, matching GFW). Gross means no regrowth is subtracted, so never present it as net.

In [ ]:
(trend["emissions_t"] / 1e6).plot.bar(title="Gross emissions from tree cover loss (Mt CO2e)", color="#7d6a5a")
plt.show()
print(f"{trend['emissions_t'].sum() / 1e9:.2f} Gt CO2e, 2001–2025")